# Notebook 4: Regression Training

This notebook trains all 24 regression experiments:
- 4 feature types: de_LDS, de_movingAve, psd_LDS, psd_movingAve
- 6 models: MLP, CNN Single, CNN Multi, EEGNet, DE-CNN, Transformer
- Task: Predict continuous PERCLOS values [0, 1]
- Loss: MSE Loss
- Metrics: MSE, RMSE, MAE, R²

All models are saved to `Models/baselines/regression/`

## 1. Import Dependencies and Check GPU

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import scipy.io as sio
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
CUDA version: 12.8


## 2. Load Model Definitions from Notebook 2

In [2]:
# Run Notebook 2 to load all model classes, datasets, and utility functions
%run 2_baseline_models.ipynb

print("Successfully loaded model definitions from Notebook 2")

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
CUDA Version: 12.8
Dataset classes defined successfully!
MLPBaseline defined!
CNN2DSingleFrame defined!
CNN2DMultiFrame defined!
EEGNet defined!
DECNN defined!
ChannelTransformer defined!

Testing All Models

MLP Baseline:
  Input shape:  torch.Size([4, 85])
  Output shape: torch.Size([4, 2])
  Total params: 21,410
  Trainable:    21,410
  ✓ Test passed!

CNN2D Single Frame:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 23,298
  Trainable:    23,298
  ✓ Test passed!

CNN2D Multi-Frame (T=5):
  Input shape:  torch.Size([4, 5, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 101,378
  Trainable:    101,378
  ✓ Test passed!

EEGNet:
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])
  Total params: 730
  Trainable:    730
  ✓ Test passed!

DE-CNN (VIGNet-style):
  Input shape:  torch.Size([4, 17, 5])
  Output shape: torch.Size([4, 2])


## 3. Configuration

In [3]:
# Paths
BASE_DIR = Path('/home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/SEED-VIG')
SAVE_DIR = BASE_DIR / 'Models' / 'baselines' / 'regression'
RESULTS_DIR = BASE_DIR / 'results' / 'baselines'

# Create directories for each feature type
for feature_key in ['de_LDS', 'de_movingAve', 'psd_LDS', 'psd_movingAve']:
    (SAVE_DIR / feature_key).mkdir(parents=True, exist_ok=True)

# Create results directory
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Global settings
TRAIN_RATIO = 0.80
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model hyperparameters
CONFIG = {
    'mlp': {
        'batch_size': 256,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 50
    },
    'cnn_single': {
        'batch_size': 128,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 50
    },
    'cnn_multi': {
        'batch_size': 64,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 50,
        'T_seq': 5,
        'step': 2
    },
    'eegnet': {
        'batch_size': 128,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 100
    },
    'decnn': {
        'batch_size': 64,
        'lr': 5e-4,
        'weight_decay': 1e-4,
        'epochs': 75
    },
    'transformer': {
        'batch_size': 64,
        'lr': 5e-4,
        'weight_decay': 1e-4,
        'epochs': 75
    }
}

print(f"Device: {DEVICE}")
print(f"Save directory: {SAVE_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Device: cuda
Save directory: /home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/SEED-VIG/Models/baselines/regression
Results directory: /home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/SEED-VIG/results/baselines


## 4. Training and Evaluation Functions

In [4]:
def evaluate_regression(model, data_loader, device):
    """
    Evaluate regression model on a dataset.
    
    Returns:
        dict with keys: mse, rmse, mae, r2
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            outputs = model(xb).squeeze()  # (batch_size,)
            
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(yb.numpy())
    
    # Concatenate all batches
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    
    # Compute metrics
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2)
    }


def train_regression(model, train_loader, val_loader, test_loader, num_epochs, lr, weight_decay, checkpoint_path, device):
    """
    Train regression model with proper train/val/test split.
    Model selection on validation set, final evaluation on test set.
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_mse = float('inf')
    
    for epoch in range(1, num_epochs + 1):
        # Training
        model.train()
        train_loss = 0.0
        
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            outputs = model(xb).squeeze()  # (batch_size,)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation (for model selection)
        val_metrics = evaluate_regression(model, val_loader, device)
        
        # Save best model based on validation MSE
        if val_metrics['mse'] < best_mse:
            best_mse = val_metrics['mse']
            torch.save(model.state_dict(), checkpoint_path)
        
        # Print progress every 10 epochs
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{num_epochs}: Train Loss={train_loss:.4f}, Val MSE={val_metrics['mse']:.4f}, RMSE={val_metrics['rmse']:.4f}, R²={val_metrics['r2']:.4f}")
    
    # Load best model and evaluate on test set (final evaluation only)
    model.load_state_dict(torch.load(checkpoint_path))
    test_metrics = evaluate_regression(model, test_loader, device)
    
    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return test_metrics

print("Training and evaluation functions defined")

Training and evaluation functions defined


## 5. Model Factory Function

In [5]:
def create_model(model_name, num_outputs=1):
    """
    Create model instance.
    
    For regression: num_outputs=1 (single continuous value)
    """
    if model_name == 'mlp':
        return MLPBaseline(input_size=85, num_outputs=num_outputs, dropout=0.3)
    
    elif model_name == 'cnn_single':
        return CNN2DSingleFrame(num_outputs=num_outputs, dropout=0.3)
    
    elif model_name == 'cnn_multi':
        return CNN2DMultiFrame(T_seq=5, num_outputs=num_outputs, dropout=0.3)
    
    elif model_name == 'eegnet':
        return EEGNet(num_outputs=num_outputs, dropout=0.25)
    
    elif model_name == 'decnn':
        return DECNN(num_outputs=num_outputs, dropout=0.3)
    
    elif model_name == 'transformer':
        return ChannelTransformer(num_outputs=num_outputs, dropout=0.3)
    
    else:
        raise ValueError(f"Unknown model: {model_name}")

print("Model factory function defined")

Model factory function defined


## 6. Single Experiment Runner

In [6]:
def run_single_regression_experiment(feature_key, model_name):
    """
    Run one complete regression experiment with 80-10-10 split.
    """
    print(f"\n{'='*80}")
    print(f"Feature: {feature_key} | Model: {model_name}")
    print(f"{'='*80}")
    
    cfg = CONFIG[model_name]
    
    # 1. Load data
    print("Loading data...")
    X_all, y_all, subj_all = load_seedvig_5band_all_features(BASE_DIR, feature_key)
    print(f"  Data shape: X={X_all.shape}, y={y_all.shape}")
    print(f"  PERCLOS range: [{y_all.min():.3f}, {y_all.max():.3f}]")
    
    # 2. Normalize features (subject-wise Z-score)
    print("Normalizing features (subject-wise)...")
    X_all = normalize_features_subjectwise(X_all, subj_all)
    
    # 3. No label binarization for regression
    
    # 4. Subject-wise split (80-10-10)
    print(f"Splitting subjects (80-10-10)...")
    train_mask, val_mask, test_mask = subject_wise_split(subj_all, train_ratio=0.80, val_ratio=0.10, seed=SEED)
    
    X_train, X_val, X_test = X_all[train_mask], X_all[val_mask], X_all[test_mask]
    y_train, y_val, y_test = y_all[train_mask], y_all[val_mask], y_all[test_mask]
    subj_train, subj_val, subj_test = subj_all[train_mask], subj_all[val_mask], subj_all[test_mask]
    
    # 5. Create datasets
    flatten = (model_name == 'mlp')
    is_multi_frame = (model_name == 'cnn_multi')
    
    if is_multi_frame:
        T_seq = cfg['T_seq']
        step = cfg['step']
        print(f"Creating multi-frame datasets (T_seq={T_seq}, step={step})...")
        
        train_ds = SEEDVIGMultiFrameDataset(X_train, y_train, subj_train, T_seq=T_seq, step=step, regression=True)
        val_ds = SEEDVIGMultiFrameDataset(X_val, y_val, subj_val, T_seq=T_seq, step=step, regression=True)
        test_ds = SEEDVIGMultiFrameDataset(X_test, y_test, subj_test, T_seq=T_seq, step=step, regression=True)
    else:
        print("Creating single-frame datasets...")
        train_ds = SEEDVIGSingleFrameDataset(X_train, y_train, flatten=flatten, regression=True)
        val_ds = SEEDVIGSingleFrameDataset(X_val, y_val, flatten=flatten, regression=True)
        test_ds = SEEDVIGSingleFrameDataset(X_test, y_test, flatten=flatten, regression=True)
    
    print(f"  Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} samples")
    
    # 6. Create dataloaders (num_workers=0 for notebook safety)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=0)
    
    # 7. Create model
    print(f"Creating {model_name} model...")
    model = create_model(model_name, num_outputs=1)  # Regression: single output
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params: {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")
    
    # 8. Train model
    checkpoint_path = SAVE_DIR / feature_key / f"{model_name}_best.pth"
    print(f"Training for {cfg['epochs']} epochs (lr={cfg['lr']}, wd={cfg['weight_decay']})...")
    
    metrics = train_regression(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_epochs=cfg['epochs'],
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
        checkpoint_path=checkpoint_path,
        device=DEVICE
    )
    
    print(f"\nFinal Test Metrics:")
    print(f"  MSE:  {metrics['mse']:.4f}")
    print(f"  RMSE: {metrics['rmse']:.4f}")
    print(f"  MAE:  {metrics['mae']:.4f}")
    print(f"  R²:   {metrics['r2']:.4f}")
    print(f"\nModel saved to: {checkpoint_path}")
    
    result = {
        'feature_key': feature_key,
        'model_name': model_name,
        'task': 'regression',
        'train_samples': len(train_ds),
        'val_samples': len(val_ds),
        'test_samples': len(test_ds),
        'num_params': total_params,
        'num_epochs': cfg['epochs'],
        'batch_size': cfg['batch_size'],
        'lr': cfg['lr'],
        'weight_decay': cfg['weight_decay'],
        'metrics': metrics,
        'checkpoint_path': str(checkpoint_path)
    }
    
    return result

print("Single experiment runner defined")

Single experiment runner defined


## 7. Run All Experiments

In [7]:
# Feature types and models
FEATURE_KEYS = ['de_LDS', 'de_movingAve', 'psd_LDS', 'psd_movingAve']
MODEL_NAMES = ['mlp', 'cnn_single', 'cnn_multi', 'eegnet', 'decnn', 'transformer']

# Run all experiments
all_results = []

print(f"\n{'#'*80}")
print(f"# STARTING ALL REGRESSION EXPERIMENTS")
print(f"# Total: {len(FEATURE_KEYS)} features × {len(MODEL_NAMES)} models = {len(FEATURE_KEYS) * len(MODEL_NAMES)} experiments")
print(f"{'#'*80}\n")

experiment_count = 0

for feature_key in FEATURE_KEYS:
    for model_name in MODEL_NAMES:
        experiment_count += 1
        print(f"\n[Experiment {experiment_count}/{len(FEATURE_KEYS) * len(MODEL_NAMES)}]")
        
        try:
            result = run_single_regression_experiment(feature_key, model_name)
            all_results.append(result)
        except Exception as e:
            print(f"ERROR in {feature_key} + {model_name}: {e}")
            import traceback
            traceback.print_exc()

print(f"\n{'#'*80}")
print(f"# ALL EXPERIMENTS COMPLETED")
print(f"# Successful: {len(all_results)}/{len(FEATURE_KEYS) * len(MODEL_NAMES)}")
print(f"{'#'*80}\n")


################################################################################
# STARTING ALL REGRESSION EXPERIMENTS
# Total: 4 features × 6 models = 24 experiments
################################################################################


[Experiment 1/24]

Feature: de_LDS | Model: mlp
Loading data...
Loaded 23 sessions (each treated as separate subject)
  X shape: (20355, 17, 5)
  y shape: (20355,)
  Unique sessions: 23
  Data shape: X=(20355, 17, 5), y=(20355,)
  PERCLOS range: [0.022, 1.000]
Normalizing features (subject-wise)...
Normalization complete (subject-wise z-score)
Splitting subjects (80-10-10)...
Subject split (80-10-10):
  Train: 18 subjects ([19, 11, 18, 23, 17, 16, 8, 7, 10, 4, 1, 20, 13, 6, 12, 15, 22, 3]), 15930 samples
  Val:   2 subjects ([5, 21]), 1770 samples
  Test:  3 subjects ([2, 14, 9]), 2655 samples
Creating single-frame datasets...
  Train: 15930, Val: 1770, Test: 2655 samples
Creating mlp model...
  Total params: 21,377
  Trainable params: 21,

## 8. Save Results

In [8]:
# Save to JSON
results_json_path = RESULTS_DIR / 'regression_results.json'
with open(results_json_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to: {results_json_path}")

Results saved to: /home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/SEED-VIG/results/baselines/regression_results.json


## 9. Generate Summary Table

In [9]:
# Convert to DataFrame for better visualization
summary_data = []

for res in all_results:
    summary_data.append({
        'Feature': res['feature_key'],
        'Model': res['model_name'],
        'Params': res['num_params'],
        'MSE': res['metrics']['mse'],
        'RMSE': res['metrics']['rmse'],
        'MAE': res['metrics']['mae'],
        'R²': res['metrics']['r2'],
        'Train Samples': res['train_samples'],
        'Test Samples': res['test_samples']
    })

df_summary = pd.DataFrame(summary_data)

# Save to CSV
csv_path = RESULTS_DIR / 'regression_summary.csv'
df_summary.to_csv(csv_path, index=False)
print(f"Summary CSV saved to: {csv_path}")

# Display summary
print("\n" + "="*100)
print("REGRESSION RESULTS SUMMARY")
print("="*100)
print(df_summary.to_string(index=False))
print("="*100)

Summary CSV saved to: /home/vortex/CSE 465/SFM1/Project/CSE465_EEG-20251204T160245Z-3-001/CSE465_EEG/SEED-VIG/results/baselines/regression_summary.csv

REGRESSION RESULTS SUMMARY
      Feature       Model  Params      MSE     RMSE      MAE       R²  Train Samples  Test Samples
       de_LDS         mlp   21377 0.033595 0.183290 0.142595 0.487266          15930          2655
       de_LDS  cnn_single   23233 0.039722 0.199303 0.157532 0.393760          15930          2655
       de_LDS   cnn_multi  101313 0.032454 0.180151 0.140123 0.504134           7938          1323
       de_LDS      eegnet     713 0.032746 0.180960 0.141725 0.500219          15930          2655
       de_LDS       decnn  117953 0.031827 0.178402 0.134552 0.514249          15930          2655
       de_LDS transformer   70529 0.031293 0.176898 0.141313 0.522401          15930          2655
 de_movingAve         mlp   21377 0.032566 0.180459 0.140759 0.502980          15930          2655
 de_movingAve  cnn_single   2

## 10. Best Model per Feature Type

In [10]:
print("\n" + "="*80)
print("BEST MODEL PER FEATURE TYPE (Lowest MSE)")
print("="*80)

for feature_key in FEATURE_KEYS:
    feature_results = df_summary[df_summary['Feature'] == feature_key]
    best_idx = feature_results['MSE'].idxmin()
    best_row = feature_results.loc[best_idx]
    
    print(f"\n{feature_key}:")
    print(f"  Best Model: {best_row['Model']}")
    print(f"  MSE:  {best_row['MSE']:.4f}")
    print(f"  RMSE: {best_row['RMSE']:.4f}")
    print(f"  MAE:  {best_row['MAE']:.4f}")
    print(f"  R²:   {best_row['R²']:.4f}")
    print(f"  Params: {best_row['Params']:,}")

print("\n" + "="*80)


BEST MODEL PER FEATURE TYPE (Lowest MSE)

de_LDS:
  Best Model: transformer
  MSE:  0.0313
  RMSE: 0.1769
  MAE:  0.1413
  R²:   0.5224
  Params: 70,529

de_movingAve:
  Best Model: cnn_single
  MSE:  0.0322
  RMSE: 0.1794
  MAE:  0.1420
  R²:   0.5086
  Params: 23,233

psd_LDS:
  Best Model: transformer
  MSE:  0.0437
  RMSE: 0.2092
  MAE:  0.1704
  R²:   0.3324
  Params: 70,529

psd_movingAve:
  Best Model: transformer
  MSE:  0.0441
  RMSE: 0.2099
  MAE:  0.1740
  R²:   0.3277
  Params: 70,529



## Summary

This notebook trained all 24 regression experiments and saved:
- Model checkpoints to `Models/baselines/regression/`
- Results JSON to `results/baselines/regression_results.json`
- Summary CSV to `results/baselines/regression_summary.csv`

Next step: Run Notebook 5 to analyze and compare all results.